In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
from google.colab import files
import io

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding="latin1")

print("File loaded:", file_name)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("Missing Values"))
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

In [ ]:
df.columns = [str(c).strip() for c in df.columns]
df = df.drop_duplicates().copy()

def find_column(names, columns):
    lookup = {str(c).lower(): c for c in columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    for c in columns:
        for name in names:
            if name.lower() in str(c).lower():
                return c
    return None

age_col = find_column(["Age"], df.columns)
income_col = find_column(["Annual Income (k$)", "Annual Income", "Income", "AnnualIncome"], df.columns)
spending_col = find_column(["Spending Score (1-100)", "Spending Score", "SpendingScore"], df.columns)

print("Age:", age_col)
print("Annual Income:", income_col)
print("Spending Score:", spending_col)

if any(c is None for c in [age_col, income_col, spending_col]):
    raise ValueError("Required Age, Annual Income, or Spending Score column was not found.")

for c in [age_col, income_col, spending_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=[age_col, income_col, spending_col]).copy()
print("Cleaned shape:", df.shape)
display(df.head())

In [ ]:
features = df[[age_col, income_col, spending_col]].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(features[age_col], kde=True, ax=axes[0])
axes[0].set_title("Age Distribution")
sns.histplot(features[income_col], kde=True, ax=axes[1])
axes[1].set_title("Annual Income Distribution")
sns.histplot(features[spending_col], kde=True, ax=axes[2])
axes[2].set_title("Spending Score Distribution")
plt.tight_layout()
plt.show()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
display(pd.DataFrame(X_scaled, columns=features.columns).head())

In [ ]:
inertia, silhouette_scores = [], []
k_values = range(2, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertia.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_values), inertia, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of Clusters")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(k_values), silhouette_scores, marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of Clusters")
axes[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

score_table = pd.DataFrame({"K": list(k_values), "Inertia": inertia, "Silhouette Score": silhouette_scores})
display(score_table)

best_k = int(score_table.loc[score_table["Silhouette Score"].idxmax(), "K"])
print("Best K by silhouette score:", best_k)

n_clusters = 4
print("K selected for requested customer segments:", n_clusters)

In [ ]:
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

print("Cluster counts:")
display(df["Cluster"].value_counts().sort_index().to_frame("Customers"))
print("Final silhouette score:", round(silhouette_score(X_scaled, df["Cluster"]), 4))

cluster_profile = df.groupby("Cluster")[[age_col, income_col, spending_col]].mean().round(2)
cluster_profile["Customer_Count"] = df.groupby("Cluster").size()
cluster_profile["Customer_Percentage"] = (cluster_profile["Customer_Count"] / len(df) * 100).round(2)
display(cluster_profile)

In [ ]:
# Label clusters from their measured income and spending behavior
ranked = cluster_profile.sort_values([spending_col, income_col], ascending=[False, False]).index.tolist()
labels = ["High Spenders", "Loyal Customers", "Discount Seekers", "Low Engagement Users"]
segment_labels = dict(zip(ranked, labels))
df["Customer_Segment"] = df["Cluster"].map(segment_labels)

segment_profile = df.groupby("Customer_Segment")[[age_col, income_col, spending_col]].mean().round(2)
segment_profile["Customer_Count"] = df.groupby("Customer_Segment").size()
segment_profile["Customer_Percentage"] = (segment_profile["Customer_Count"] / len(df) * 100).round(2)
display(segment_profile)

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x=income_col, y=spending_col,
                hue="Customer_Segment", style="Customer_Segment", s=80)
plt.title("Customer Segmentation: Income vs Spending Score")
plt.xlabel("Annual Income")
plt.ylabel("Spending Score")
plt.legend(title="Customer Segment", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x=age_col, y=spending_col,
                hue="Customer_Segment", style="Customer_Segment", s=80)
plt.title("Customer Segmentation: Age vs Spending Score")
plt.xlabel("Age")
plt.ylabel("Spending Score")
plt.legend(title="Customer Segment", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame({"PC1": X_pca[:,0], "PC2": X_pca[:,1],
                       "Customer_Segment": df["Customer_Segment"].values})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2",
                hue="Customer_Segment", style="Customer_Segment", s=80)
plt.title("Customer Segmentation — PCA View")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(title="Customer Segment", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_data = segment_profile[[age_col, income_col, spending_col]]
heatmap_scaled = pd.DataFrame(
    StandardScaler().fit_transform(heatmap_data),
    index=heatmap_data.index,
    columns=heatmap_data.columns
)

plt.figure(figsize=(10, 5))
sns.heatmap(heatmap_scaled, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Customer Segment Profile Heatmap")
plt.xlabel("Customer Attributes")
plt.ylabel("Customer Segment")
plt.tight_layout()
plt.show()

In [ ]:
segment_counts = df["Customer_Segment"].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=segment_counts.index, y=segment_counts.values)
plt.title("Number of Customers by Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()
display(segment_counts.to_frame("Customer_Count"))

In [ ]:
recommendations = {
    "High Spenders": "Offer premium products, exclusive memberships, personalized offers, and early access to new products.",
    "Loyal Customers": "Use loyalty programs, repeat-purchase rewards, personalized recommendations, and retention campaigns.",
    "Discount Seekers": "Use targeted discounts, bundles, seasonal promotions, and price-sensitive offers.",
    "Low Engagement Users": "Run re-engagement campaigns, personalized reminders, introductory offers, and feedback surveys."
}

print("BUSINESS RECOMMENDATIONS")
print("=" * 60)
for segment in segment_profile.index:
    print(f"\n{segment}:")
    print(recommendations[segment])

In [ ]:
print("FINAL CUSTOMER SEGMENTATION SUMMARY")
print("=" * 60)
for segment, row in segment_profile.iterrows():
    print(f"\n{segment}")
    print(f"Customers: {int(row['Customer_Count']):,}")
    print(f"Share: {row['Customer_Percentage']:.2f}%")
    print(f"Average Age: {row[age_col]:.2f}")
    print(f"Average Annual Income: {row[income_col]:.2f}")
    print(f"Average Spending Score: {row[spending_col]:.2f}")

print("\nNumber of clusters:", n_clusters)
print("Final silhouette score:", round(silhouette_score(X_scaled, df["Cluster"]), 4))